# Sniper Pullback ML — EDA & Full Pipeline
**Objetivo:** añadir capa ML al bot Binance Futures para pasar de WR 58% → >70%

Pipeline: descarga → EDA → feature engineering → entrenamiento LightGBM/XGBoost → calibración → backtest ML

## 0 — Setup & Librerías

In [ ]:
import sys, os, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
%matplotlib inline

ROOT      = os.path.abspath('..')
DATA_DIR  = os.path.join(ROOT, 'data')
MODEL_DIR = os.path.join(ROOT, 'models')
SCRIPTS   = os.path.join(ROOT, 'scripts')
sys.path.insert(0, SCRIPTS)

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'LightGBM {lgb.__version__}  |  XGBoost {xgb.__version__}  |  pandas {pd.__version__}')
print(f'DATA_DIR  = {DATA_DIR}')
print(f'MODEL_DIR = {MODEL_DIR}')

In [ ]:
# ── Parámetros Bot (deben coincidir con binance_bot_django.py) ─────────────────
DURACION_VELAS    = 15      # velas de hold
ADX_MIN           = 28
EMA50_SLOPE_N     = 5
EMA50_SLOPE_MIN   = 0.03
RSI_PULLBACK_CALL = 42
RSI_PULLBACK_PUT  = 58
RSI_RESUME_CALL   = 50
RSI_RESUME_PUT    = 50
VOL_FILTER_RATIO  = 0.8
STAKE             = 1.0
PAYOUT            = 0.95
WARMUP_CANDLES    = 60
SIMBOLOS          = ['ETHUSDT', 'BTCUSDT', 'SOLUSDT', 'BNBUSDT']
INTERVALOS        = ['1m', '5m', '15m']
DIAS              = 180

BASE_URL = 'https://api.binance.com/api/v3/klines'
COLS_11  = ['open_time','open','high','low','close','volume','close_time',
            'quote_vol','num_trades','taker_buy_base','taker_buy_quote']
print('Parámetros configurados OK')

## 1 — Descarga de Datos

In [ ]:
def descargar_historico(simbolo, intervalo, dias, output_csv):
    end_ms   = int(time.time() * 1000)
    start_ms = end_ms - dias * 86_400_000
    current  = start_ms
    all_rows = []
    print(f'  Descargando {simbolo} {intervalo} ({dias}d)...', end=' ', flush=True)
    while current < end_ms:
        url = f'{BASE_URL}?symbol={simbolo}&interval={intervalo}&startTime={current}&limit=1000'
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=30) as r:
                data = json.loads(r.read())
            if not data: break
            all_rows.extend(data)
            current = data[-1][0] + 1
            time.sleep(0.1)
        except Exception as e:
            print(f'\n    Retry ({e})'); time.sleep(8)
    os.makedirs(os.path.dirname(output_csv) or '.', exist_ok=True)
    header = ','.join(['open_time','open','high','low','close','volume','close_time',
                       'quote_vol','num_trades','taker_buy_base','taker_buy_quote','ignore'])
    with open(output_csv, 'w') as f:
        f.write(header + '\n')
        for row in all_rows: f.write(','.join(str(x) for x in row) + '\n')
    print(f'{len(all_rows):,} velas OK')
    return len(all_rows)

# Descarga todo (saltea si ya existe)
for sym in SIMBOLOS:
    for iv in INTERVALOS:
        path = os.path.join(DATA_DIR, f'{sym}_{iv}.csv')
        if os.path.exists(path):
            rows = sum(1 for _ in open(path)) - 1
            print(f'  {sym} {iv}: ya existe ({rows:,} filas)')
        else:
            descargar_historico(sym, iv, DIAS, path)

In [ ]:
def load_csv(path):
    df = pd.read_csv(path, usecols=range(11))
    df.columns = COLS_11
    for c in ['open','high','low','close','volume','quote_vol','taker_buy_base','taker_buy_quote']:
        df[c] = df[c].astype(float)
    df['num_trades'] = df['num_trades'].astype(int)
    df['open_time']  = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    return df.set_index('open_time').sort_index()

eth_1m  = load_csv(os.path.join(DATA_DIR, 'ETHUSDT_1m.csv'))
eth_5m  = load_csv(os.path.join(DATA_DIR, 'ETHUSDT_5m.csv'))
eth_15m = load_csv(os.path.join(DATA_DIR, 'ETHUSDT_15m.csv'))

print(f'ETH 1m : {len(eth_1m):,} filas  |  {eth_1m.index[0].date()} → {eth_1m.index[-1].date()}')
print(f'ETH 5m : {len(eth_5m):,} filas')
print(f'ETH 15m: {len(eth_15m):,} filas')
print(f'\nNaNs en 1m:\n{eth_1m.isnull().sum()}')
assert len(eth_1m) >= 200_000, f'Solo {len(eth_1m):,} filas — mín 200k'
print('\n✓ Datos suficientes')

## 2 — EDA Básico

In [ ]:
# Gaps > 2 minutos
gaps     = eth_1m.index.to_series().diff().dt.total_seconds().dropna()
big_gaps = gaps[gaps > 120]
print(f'Gaps > 2 min en ETH 1m: {len(big_gaps)}')
if len(big_gaps) > 0:
    print(big_gaps.sort_values(ascending=False).head(10))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
eth_1m['close'].plot(ax=axes[0], title='ETH/USDT — Close 180d', lw=0.5)
axes[0].set_ylabel('Precio USD')
eth_1m['volume'].rolling(60).mean().plot(ax=axes[1], title='Volumen MA60 1m', lw=0.8, color='orange')
axes[1].set_ylabel('Vol ETH')
plt.tight_layout(); plt.show()

In [ ]:
# Helpers vectorizados
def ema(s, n): return s.ewm(span=n, adjust=False).mean()

def rsi_vec(s, n=14):
    d = s.diff(); g = d.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    l = (-d.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100 / (1 + g / l.replace(0, np.nan))

def atr_vec(h, l, c, n=14):
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/n, adjust=False).mean()

def adx_vec(h, l, c, n=14):
    up = h.diff(); dn = -l.diff()
    dmp = np.where((up>dn)&(up>0), up, 0.0)
    dmm = np.where((dn>up)&(dn>0), dn, 0.0)
    a   = atr_vec(h, l, c, n)
    dip = 100 * pd.Series(dmp, index=c.index).ewm(alpha=1/n, adjust=False).mean() / a
    dim = 100 * pd.Series(dmm, index=c.index).ewm(alpha=1/n, adjust=False).mean() / a
    dx  = 100 * (dip-dim).abs() / (dip+dim).replace(0, np.nan)
    return dx.ewm(alpha=1/n, adjust=False).mean()

def merge_htf(df_1m, df_htf, suffix):
    right = df_htf.copy()
    right.columns = [f'{c}_{suffix}' for c in right.columns]
    return pd.merge_asof(
        df_1m.reset_index(),
        right.reset_index().rename(columns={'open_time': f'ot_{suffix}'}),
        left_on='open_time', right_on=f'ot_{suffix}', direction='backward',
    ).set_index('open_time').drop(columns=[f'ot_{suffix}'], errors='ignore')

# Merge 1m + 5m + 15m
df = merge_htf(eth_1m, eth_5m,  '5m')
df = merge_htf(df,    eth_15m, '15m')
print(f'df merged: {df.shape}  cols: {list(df.columns[:8])}')

In [ ]:
# Señales Rule-based
rsi_1m      = rsi_vec(df['close'], 14)
rsi_ant     = rsi_1m.shift(1)
adx_5m_v    = adx_vec(df['high_5m'], df['low_5m'], df['close_5m'], 14)
ema50_15m_v = ema(df['close_15m'], 50)
slope_pct   = (ema50_15m_v - ema50_15m_v.shift(EMA50_SLOPE_N)) / ema50_15m_v.shift(EMA50_SLOPE_N).replace(0,np.nan) * 100
ema21_v     = ema(df['close'], 21)
vol_ma20    = df['volume'].rolling(20).mean()

bull_macro = (slope_pct >= EMA50_SLOPE_MIN) & (df['close'] > ema50_15m_v)
bear_macro = (slope_pct <= -EMA50_SLOPE_MIN) & (df['close'] < ema50_15m_v)
pull_call  = (rsi_ant < RSI_PULLBACK_CALL) & (rsi_1m > RSI_RESUME_CALL)
pull_put   = (rsi_ant > RSI_PULLBACK_PUT)  & (rsi_1m < RSI_RESUME_PUT)
vol_ok     = df['volume'] >= vol_ma20 * VOL_FILTER_RATIO

call_mask = (adx_5m_v >= ADX_MIN) & bull_macro & pull_call & (df['close']>df['open']) & (df['close']>=ema21_v) & vol_ok
put_mask  = (adx_5m_v >= ADX_MIN) & bear_macro & pull_put  & (df['close']<df['open']) & (df['close']<=ema21_v) & vol_ok

fut_close  = df['close'].shift(-DURACION_VELAS)
label_call = (fut_close > df['close']).astype(int)
label_put  = (fut_close < df['close']).astype(int)

df_calls = df[call_mask].copy(); df_calls['label'] = label_call[call_mask]
df_puts  = df[put_mask].copy();  df_puts['label']  = label_put[put_mask]
df_calls = df_calls.iloc[WARMUP_CANDLES:-DURACION_VELAS]
df_puts  = df_puts.iloc[WARMUP_CANDLES:-DURACION_VELAS]

total_dias = (df.index[-1] - df.index[0]).days
print(f'CALL: {len(df_calls):,} señales  WR={df_calls["label"].mean()*100:.1f}%  ops/día={len(df_calls)/total_dias:.1f}')
print(f'PUT : {len(df_puts):,} señales   WR={df_puts["label"].mean()*100:.1f}%  ops/día={len(df_puts)/total_dias:.1f}')

In [ ]:
# WR por hora UTC y día de semana
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

vc = df_calls['label'].value_counts()
axes[0,0].bar(['LOSS (0)','WIN (1)'], [vc.get(0,0), vc.get(1,0)], color=['#e74c3c','#2ecc71'])
axes[0,0].set_title('CALL — WIN vs LOSS')

wr_hora = df_calls.groupby(df_calls.index.hour)['label'].agg(['mean','count'])
colors  = ['#2ecc71' if x>=0.60 else '#e74c3c' if x<0.50 else '#f39c12' for x in wr_hora['mean']]
axes[0,1].bar(wr_hora.index, wr_hora['mean']*100, color=colors)
axes[0,1].axhline(50, ls='--', c='k', lw=1)
axes[0,1].axhline(df_calls['label'].mean()*100, ls=':', c='blue', lw=1.5, label='WR global')
axes[0,1].set_title('WR CALL por hora UTC'); axes[0,1].legend()

days = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
wr_dow = df_calls.groupby(df_calls.index.dayofweek)['label'].mean()
axes[1,0].bar([days[i] for i in wr_dow.index], wr_dow*100,
              color=['#2ecc71' if x>0.58 else '#e74c3c' for x in wr_dow])
axes[1,0].axhline(50, ls='--', c='k'); axes[1,0].set_title('WR CALL por día')

wr_hora_p = df_puts.groupby(df_puts.index.hour)['label'].agg(['mean','count'])
axes[1,1].bar(wr_hora.index, wr_hora['count'],   color='#3498db', alpha=0.7, label='CALL')
axes[1,1].bar(wr_hora_p.index, wr_hora_p['count'], color='#e74c3c', alpha=0.5, label='PUT')
axes[1,1].set_title('# Señales por hora UTC'); axes[1,1].legend()

plt.tight_layout(); plt.show()
print(f"Horas con WR > 60%: {list(wr_hora[wr_hora['mean']>0.60].index)}")
print(f"Horas con WR < 50%: {list(wr_hora[wr_hora['mean']<0.50].index)}")

In [ ]:
# RSI / ADX / Slope: distribución WIN vs LOSS
df_calls['rsi']   = rsi_1m[call_mask].values[:len(df_calls)]
df_calls['adx']   = adx_5m_v[call_mask].values[:len(df_calls)]
df_calls['slope'] = slope_pct[call_mask].values[:len(df_calls)]

from scipy.stats import ks_2samp
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feat, title in zip(axes, ['rsi','adx','slope'], ['RSI 1m','ADX 5m','EMA50 slope%']):
    wins   = df_calls[df_calls['label']==1][feat].dropna()
    losses = df_calls[df_calls['label']==0][feat].dropna()
    ax.hist(wins,   bins=40, alpha=0.6, label=f'WIN  n={len(wins)}',  color='#2ecc71', density=True)
    ax.hist(losses, bins=40, alpha=0.6, label=f'LOSS n={len(losses)}', color='#e74c3c', density=True)
    stat, p = ks_2samp(wins, losses)
    ax.set_title(f'{title}  KS_p={p:.3f}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Volatilidad rolling y régimen
log_ret  = np.log(eth_1m['close'] / eth_1m['close'].shift(1)).dropna()
roll_vol = log_ret.rolling(60).std() * np.sqrt(1440)
vol_med  = float(roll_vol.median())

df_calls_r = df_calls.copy()
df_calls_r['regime'] = np.where(roll_vol.reindex(df_calls_r.index, method='ffill') > vol_med,
                                'HIGH_VOL', 'LOW_VOL')
wr_r = df_calls_r.groupby('regime')['label'].agg(['mean','count'])
print('WR por régimen (CALL):')
print(wr_r.rename(columns={'mean':'WR','count':'N'}).to_string())

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
eth_1m['close'].plot(ax=axes[0], lw=0.4, title='ETH Close + régimen vol')
axes[0].fill_between(roll_vol.index, eth_1m['close'].iloc[1:].min(), eth_1m['close'].max(),
    where=roll_vol.values > vol_med, alpha=0.15, color='red', label='HIGH-VOL')
axes[0].legend()
roll_vol.plot(ax=axes[1], lw=0.8, color='orange', title='Volatilidad rolling 60m')
axes[1].axhline(vol_med, ls='--', c='red', label=f'mediana={vol_med:.4f}'); axes[1].legend()
plt.tight_layout(); plt.show()

## 3 — Feature Engineering

In [ ]:
from feature_engineering import build_features

EXCLUDE_COLS = {
    'open','high','low','close','volume','close_time','quote_vol','num_trades',
    'taker_buy_base','taker_buy_quote',
    'open_5m','high_5m','low_5m','close_5m','volume_5m','close_time_5m',
    'quote_vol_5m','num_trades_5m','taker_buy_base_5m','taker_buy_quote_5m',
    'open_15m','high_15m','low_15m','close_15m','volume_15m','close_time_15m',
    'quote_vol_15m','num_trades_15m','taker_buy_base_15m','taker_buy_quote_15m',
    'signal','label_call','label_put','future_ret_pct',
}

dfs = {}
for sym in SIMBOLOS:
    parquet = os.path.join(DATA_DIR, f'features_{sym}.parquet')
    if os.path.exists(parquet):
        dfs[sym] = pd.read_parquet(parquet)
        print(f'  {sym}: cargado ({len(dfs[sym]):,} filas, {dfs[sym].shape[1]} cols)')
    elif os.path.exists(os.path.join(DATA_DIR, f'{sym}_1m.csv')):
        print(f'  {sym}: calculando features...')
        dfs[sym] = build_features(sym)
        dfs[sym].to_parquet(parquet)
        print(f'        → {len(dfs[sym]):,} filas guardadas')
    else:
        print(f'  {sym}: sin CSV, saltando')

print(f'\nSímbolos disponibles: {list(dfs.keys())}')

In [ ]:
# Resumen señales por símbolo
print(f'{"SYM":12} {"CALL":>8} {"WR_C":>7} {"PUT":>8} {"WR_P":>7}')
print('-' * 50)
for sym, df in dfs.items():
    calls = df[df['signal']=='CALL']
    puts  = df[df['signal']=='PUT']
    wr_c  = calls['label_call'].mean() if len(calls)>0 else 0
    wr_p  = puts['label_put'].mean()   if len(puts)>0  else 0
    print(f'  {sym:10} {len(calls):8,} {wr_c*100:6.1f}% {len(puts):8,} {wr_p*100:6.1f}%')

In [ ]:
# Correlación de features (ETH CALL)
df_eth = dfs.get('ETHUSDT')
if df_eth is not None:
    feat_cols = [c for c in df_eth.columns if c not in EXCLUDE_COLS]
    df_sig    = df_eth[df_eth['signal']=='CALL'][feat_cols].dropna()
    corr      = df_sig.corr()

    high_corr = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
                 for i in range(len(corr.columns))
                 for j in range(i+1, len(corr.columns))
                 if abs(corr.iloc[i,j]) > 0.85]
    print(f'Pares con |r| > 0.85: {len(high_corr)}')
    for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2]))[:10]:
        print(f'  {a:30} ↔ {b:30}  r={r:.3f}')

    n = min(30, len(feat_cols))
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones((n, n), dtype=bool))
    sns.heatmap(corr.iloc[:n,:n], mask=mask, center=0, cmap='RdBu_r', vmin=-1, vmax=1,
                linewidths=0.3, ax=ax)
    ax.set_title('Correlación Features — ETH CALL (primeras 30)')
    plt.tight_layout(); plt.show()

## 4 — Baseline (Logistic + RF)

In [ ]:
def run_baseline(sym='ETHUSDT', direction='CALL'):
    label_col = f'label_{direction.lower()}'
    df        = dfs.get(sym)
    if df is None: return
    feat_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
    df_sig    = df[df['signal']==direction].copy()
    if len(df_sig) < 100: print(f'{sym}/{direction}: insuficiente'); return

    n = len(df_sig); i70, i85 = int(n*0.70), int(n*0.85)
    tr, va, te = df_sig.iloc[:i70], df_sig.iloc[i70:i85], df_sig.iloc[i85:]

    imp = SimpleImputer(strategy='median'); sc = StandardScaler()
    X_tr = sc.fit_transform(imp.fit_transform(tr[feat_cols].astype(np.float32)))
    X_va = sc.transform(imp.transform(va[feat_cols].astype(np.float32)))
    X_te = sc.transform(imp.transform(te[feat_cols].astype(np.float32)))
    y_tr, y_va, y_te = tr[label_col].values, va[label_col].values, te[label_col].values

    results = {}
    for name, model in [
        ('LogReg', LogisticRegression(class_weight='balanced', max_iter=500)),
        ('RF-100', RandomForestClassifier(100, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)),
    ]:
        model.fit(X_tr, y_tr)
        p_va = model.predict_proba(X_va)[:,1]
        auc  = roc_auc_score(y_va, p_va)
        wr   = y_va[p_va>=0.52].mean() if (p_va>=0.52).sum()>0 else 0
        print(f'  {sym}/{direction}/{name}: AUC={auc:.3f}  WR@0.52={wr*100:.1f}%  ops={int((p_va>=0.52).sum())}')
        results[name] = {'AUC': auc, 'WR@0.52': wr}
    return results

print('=== BASELINE ===' )
run_baseline('ETHUSDT', 'CALL')
run_baseline('ETHUSDT', 'PUT')

## 5 — LightGBM + XGBoost + Calibración

In [ ]:
LGBM_PARAMS = dict(
    n_estimators=1000, learning_rate=0.02, max_depth=5, num_leaves=31,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.7,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
)

def train_lgbm(X_tr, y_tr, X_va, y_va, params=None):
    m = lgb.LGBMClassifier(**(params or LGBM_PARAMS))
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    return m

def train_xgb(X_tr, y_tr, X_va, y_va):
    sw = max((y_tr==0).sum() / max((y_tr==1).sum(),1), 1.0)
    m  = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.02, max_depth=5,
                            min_child_weight=30, subsample=0.8, colsample_bytree=0.7,
                            scale_pos_weight=sw, random_state=42, n_jobs=-1,
                            eval_metric='logloss', early_stopping_rounds=50, verbosity=0)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return m

def expected_pnl(wr, n_ops=1, stake=STAKE, payout=PAYOUT):
    return n_ops * (wr * payout * stake - (1-wr) * stake)

def select_threshold(probs, labels, min_ops=5):
    best_pnl, best_thr = -999, 0.50
    rows = []
    for thr in np.arange(0.45, 0.80, 0.01):
        mask = probs >= thr
        if mask.sum() < min_ops: continue
        wr  = labels[mask].mean()
        pnl = mask.sum() * (wr*PAYOUT*STAKE - (1-wr)*STAKE)
        rows.append({'thr': round(thr,2), 'ops': mask.sum(), 'wr': wr, 'pnl': pnl})
        if pnl > best_pnl: best_pnl, best_thr = pnl, round(thr,2)
    return best_thr, best_pnl, pd.DataFrame(rows)

print('Funciones de entrenamiento definidas OK')

In [ ]:
def full_pipeline(sym='ETHUSDT', direction='CALL', verbose=True):
    label_col = f'label_{direction.lower()}'
    df        = dfs.get(sym)
    if df is None: return None
    feat_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
    df_sig    = df[df['signal']==direction].copy()
    if len(df_sig) < 100: return None

    n = len(df_sig); i70, i85 = int(n*0.70), int(n*0.85)
    tr, va, te = df_sig.iloc[:i70], df_sig.iloc[i70:i85], df_sig.iloc[i85:]
    if verbose:
        print(f'\n{"="*55}')
        print(f'  {sym}/{direction}  Train:{len(tr)} Val:{len(va)} Test:{len(te)}')
        print(f'  Periodo: {tr.index[0].date()} → {te.index[-1].date()}')

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(tr[feat_cols].astype(np.float32))
    X_va = imp.transform(va[feat_cols].astype(np.float32))
    X_te = imp.transform(te[feat_cols].astype(np.float32))
    y_tr, y_va, y_te = tr[label_col].values, va[label_col].values, te[label_col].values

    lgbm_raw = train_lgbm(X_tr, y_tr, X_va, y_va)
    lgbm_cal = CalibratedClassifierCV(lgbm_raw, method='isotonic', cv='prefit')
    lgbm_cal.fit(X_va, y_va)
    p_va_l = lgbm_cal.predict_proba(X_va)[:,1]
    auc_l  = roc_auc_score(y_va, p_va_l)

    xgb_raw = train_xgb(X_tr, y_tr, X_va, y_va)
    xgb_cal = CalibratedClassifierCV(xgb_raw, method='isotonic', cv='prefit')
    xgb_cal.fit(X_va, y_va)
    p_va_x = xgb_cal.predict_proba(X_va)[:,1]
    auc_x  = roc_auc_score(y_va, p_va_x)

    if verbose: print(f'  LGBM AUC: {auc_l:.4f}  |  XGB AUC: {auc_x:.4f}')

    best_cal  = lgbm_cal if auc_l >= auc_x else xgb_cal
    best_raw  = lgbm_raw if auc_l >= auc_x else xgb_raw
    best_probs= p_va_l   if auc_l >= auc_x else p_va_x
    best_name = 'lgbm'   if auc_l >= auc_x else 'xgb'

    thr, _, df_thr = select_threshold(best_probs, y_va)

    p_te  = best_cal.predict_proba(X_te)[:,1]
    mask  = p_te >= thr
    wr_te = y_te[mask].mean() if mask.sum()>0 else 0.0
    ops   = int(mask.sum())
    pnl   = ops * (wr_te*PAYOUT*STAKE - (1-wr_te)*STAKE)
    auc_te= roc_auc_score(y_te, p_te) if len(np.unique(y_te))>1 else 0.5

    if verbose:
        print(f'  [TEST] AUC={auc_te:.4f}  ops={ops}  WR={wr_te*100:.1f}%  P&L=${pnl:.2f}')
        flag = '✓ EXCELENTE' if wr_te>=0.70 else '✓ BUENO' if wr_te>=0.63 else '! Revisar'
        print(f'  {flag}')

    # Guardar
    tag = f'{sym}_{direction.lower()}'
    joblib.dump(best_cal, os.path.join(MODEL_DIR, f'{best_name}_{tag}.pkl'))
    joblib.dump(imp,      os.path.join(MODEL_DIR, f'imputer_{tag}.pkl'))
    with open(os.path.join(MODEL_DIR, f'threshold_{tag}.txt'), 'w') as f: f.write(str(thr))
    with open(os.path.join(MODEL_DIR, f'features_{tag}.json'), 'w') as f: json.dump(feat_cols, f)

    return dict(sym=sym, dir=direction, best=best_name, auc_val=max(auc_l,auc_x),
                thr=thr, wr_test=wr_te, ops_test=ops, pnl_test=pnl,
                feat_cols=feat_cols, df_thr=df_thr,
                model_raw=best_raw, model_cal=best_cal,
                X_va=X_va, y_va=y_va, probs_va=best_probs,
                X_te=X_te, y_te=y_te, probs_te=p_te)

results_all = {}
print('Iniciando entrenamiento...\n')
for sym in list(dfs.keys()):
    for direc in ['CALL', 'PUT']:
        key = f'{sym}_{direc}'
        res = full_pipeline(sym, direc)
        if res:
            results_all[key] = res

## 6 — Calibración y Threshold Sweep

In [ ]:
# Calibration curves
n_m = len(results_all)
if n_m > 0:
    cols_p = min(n_m, 4); rows_p = (n_m + cols_p - 1) // cols_p
    fig, axes = plt.subplots(rows_p, cols_p, figsize=(5*cols_p, 4*rows_p))
    axes_flat = np.array(axes).flatten()
    for idx, (key, r) in enumerate(results_all.items()):
        ax = axes_flat[idx]
        fp, mp = calibration_curve(r['y_va'], r['probs_va'], n_bins=10)
        ax.plot(mp, fp, 's-b', ms=4, label='Model')
        ax.plot([0,1],[0,1],'k--', label='Perfect')
        ax.set_title(f'{key}\nthr={r["thr"]}  WR_test={r["wr_test"]*100:.1f}%')
        ax.set_xlabel('P predicho'); ax.set_ylabel('WR real'); ax.legend(fontsize=8)
    for idx in range(n_m, len(axes_flat)): axes_flat[idx].set_visible(False)
    plt.suptitle('Calibration Curves (Val Set)', fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

In [ ]:
# Threshold sweep para ETH CALL + PUT
for key in ['ETHUSDT_CALL', 'ETHUSDT_PUT']:
    r = results_all.get(key)
    if not r or r['df_thr'].empty: continue
    dt = r['df_thr']
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax2 = ax1.twinx()
    ax1.plot(dt['thr'], dt['wr']*100, 'b-o', ms=4, label='WR %')
    ax2.plot(dt['thr'], dt['pnl'],    'r-s', ms=4, label='P&L $')
    ax1.axvline(r['thr'], ls=':', c='green', lw=2, label=f'optimal={r["thr"]}')
    ax1.set_ylim(40, 85); ax1.set_xlabel('Threshold')
    ax1.set_ylabel('WR %', color='b'); ax2.set_ylabel('P&L $', color='r')
    ax1.set_title(f'Threshold Sweep — {key}')
    l1,lb1 = ax1.get_legend_handles_labels(); l2,lb2 = ax2.get_legend_handles_labels()
    ax1.legend(l1+l2, lb1+lb2, loc='upper left')
    plt.tight_layout(); plt.show()

## 7 — SHAP: Importancia de Features

In [ ]:
try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False
    print('shap no instalado — pip install shap')

if SHAP_OK and 'ETHUSDT_CALL' in results_all:
    r        = results_all['ETHUSDT_CALL']
    raw      = r['model_raw']
    n_sample = min(500, len(r['X_va']))
    X_shap   = r['X_va'][:n_sample]

    explainer  = shap.TreeExplainer(raw)
    shap_vals  = explainer.shap_values(X_shap)
    if isinstance(shap_vals, list): shap_vals = shap_vals[1]

    feat_cols = r['feat_cols']
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_vals, X_shap, feature_names=feat_cols,
                      plot_type='bar', max_display=20, show=False)
    plt.title('SHAP Importance — ETH CALL'); plt.tight_layout(); plt.show()

    plt.figure(figsize=(10, 10))
    shap.summary_plot(shap_vals, X_shap, feature_names=feat_cols, max_display=20, show=False)
    plt.title('SHAP Beeswarm — ETH CALL'); plt.tight_layout(); plt.show()

    mean_abs = np.abs(shap_vals).mean(axis=0)
    top_pct  = mean_abs.max() / mean_abs.sum()
    top_feat = feat_cols[int(mean_abs.argmax())]
    print(f'Feature dominante: {top_feat} ({top_pct*100:.1f}%)')
    if top_pct > 0.30:
        print('  ⚠ Revisar posible lookahead')
    else:
        print('  ✓ Importancia distribuida correctamente')

## 8 — Tabla Resumen y Equity Curves

In [ ]:
# Tabla resumen
print(f'{"MOD":22} {"AUC_val":>8} {"thr":>6} {"WR_test":>9} {"ops":>6} {"P&L":>9}')
print('-' * 65)
top_keys = []
for key, r in sorted(results_all.items()):
    flag = '✓' if r['wr_test'] >= 0.63 else '!'
    print(f'  {key:20} {r["auc_val"]:8.4f} {r["thr"]:6.2f} '
          f'{r["wr_test"]*100:8.1f}% {r["ops_test"]:6} ${r["pnl_test"]:8.2f} {flag}')
    if r['wr_test'] >= 0.60: top_keys.append(key)

print(f'\nModelos aptos (WR ≥ 60%): {top_keys}')

In [ ]:
# Equity curve: Rule-only vs ML threshold
df_eth = dfs.get('ETHUSDT')
r      = results_all.get('ETHUSDT_CALL')

if df_eth is not None and r:
    cm     = df_eth['signal'].values == 'CALL'
    labels = df_eth['label_call'].values
    probs  = np.zeros(len(df_eth))
    sig_i  = np.where(cm)[0]
    n      = min(len(sig_i), len(r['probs_te']))
    probs[sig_i[:n]] = r['probs_te'][:n]

    fig, ax = plt.subplots(figsize=(14, 5))
    for lbl, thr, color in [
        ('Rule-only', None,  '#95a5a6'),
        ('ML thr=0.55', 0.55, '#3498db'),
        ('ML thr=0.60', 0.60, '#2ecc71'),
        ('ML thr=0.65', 0.65, '#e74c3c'),
    ]:
        mask = cm if thr is None else cm & (probs >= thr)
        lbl_arr = labels[mask]
        if len(lbl_arr) == 0: continue
        eq  = 100 + np.cumsum(np.where(lbl_arr==1, PAYOUT*STAKE, -STAKE))
        ts  = df_eth.index[mask][:len(eq)]
        ax.plot(ts, eq, label=f'{lbl} (final=${eq[-1]:.0f})', lw=1.5, color=color)

    ax.axhline(100, ls='--', c='k', lw=1)
    ax.set_title('Equity Curves — Rule-based vs ML (ETH CALL)')
    ax.set_ylabel('Capital ($)'); ax.legend(); plt.tight_layout(); plt.show()

## 9 — Drift Detection (PSI)

In [ ]:
def psi_feature(expected, actual, bins=10):
    mn = min(expected.min(), actual.min())
    mx = max(expected.max(), actual.max())
    eps = 1e-8
    bpts = np.linspace(mn, mx, bins+1)
    e = np.histogram(expected, bins=bpts)[0] / len(expected) + eps
    a = np.histogram(actual,   bins=bpts)[0] / len(actual)   + eps
    return float(np.sum((a-e)*np.log(a/e)))

r = results_all.get('ETHUSDT_CALL')
if r and 'ETHUSDT' in dfs:
    df_sig   = dfs['ETHUSDT'][dfs['ETHUSDT']['signal']=='CALL']
    n        = len(df_sig); i70 = int(n*0.70)
    feat_cols= r['feat_cols']
    tr_feats = df_sig.iloc[:i70][feat_cols]
    rc_feats = df_sig.iloc[i70:][feat_cols]

    psi_scores = {}
    for c in feat_cols:
        tr = tr_feats[c].dropna().values
        rc = rc_feats[c].dropna().values
        if len(tr)>10 and len(rc)>10:
            psi_scores[c] = psi_feature(tr, rc)

    psi_df = pd.Series(psi_scores).sort_values(ascending=False)
    print('PSI (train vs reciente) — top 15:')
    print(psi_df.head(15).to_string())
    drift_feats = list(psi_df[psi_df > 0.20].index)
    print(f'\nFeatures con drift (PSI>0.20): {drift_feats}')

    colors = ['#e74c3c' if v>0.20 else '#f39c12' if v>0.10 else '#2ecc71'
              for v in psi_df.head(15)]
    plt.figure(figsize=(10, 4))
    psi_df.head(15).plot(kind='bar', color=colors)
    plt.axhline(0.20, ls='--', c='red',    label='Drift (>0.2)')
    plt.axhline(0.10, ls=':',  c='orange', label='Warning (>0.1)')
    plt.title('PSI por Feature'); plt.legend(); plt.tight_layout(); plt.show()

## 10 — Resumen Ejecutivo y Próximos Pasos

In [ ]:
print('=' * 60)
print('RESUMEN EJECUTIVO — SNIPER PULLBACK ML')
print('=' * 60)

if results_all:
    best_key = max(results_all, key=lambda k: results_all[k]['wr_test'])
    r        = results_all[best_key]
    print(f'\n  Mejor modelo : {best_key}')
    print(f'  Algoritmo    : {r["best"]}')
    print(f'  AUC (val)    : {r["auc_val"]:.4f}')
    print(f'  Threshold    : {r["thr"]}')
    print(f'  WR (test)    : {r["wr_test"]*100:.1f}%')
    print(f'  P&L (test)   : ${r["pnl_test"]:.2f}')
    print(f'  Ops en test  : {r["ops_test"]}')

print('\nArchivos generados:')
for fn in sorted(os.listdir(MODEL_DIR)):
    sz = os.path.getsize(os.path.join(MODEL_DIR, fn))
    print(f'  models/{fn:<42}  {sz//1024} KB')

print('''
PRÓXIMOS PASOS:
  1. Si WR_test >= 63%: integrar ML en binance_bot_django.py
     → Añadir extraer_features_live() + evaluar_senal() ML gate
  2. Ejecutar 1 semana en papel (orden_real=False)
  3. Si WR live >= 62%: activar real con stake mínimo
  4. Re-entrenar cada 30 días o si PSI > 0.20 en algún feature clave
  5. Monitorear WR rolling 50 ops — alertar si cae < 55%
''')